# 07: Statistical Analysis

Full inferential analysis of results_master.csv (960 verified
run-seed combinations) including descriptive statistics, assumption
checks, two-way factorial ANOVAs for RQ1 (performance) and RQ2
(fairness), the RQ3 comparative metric sensitivity assembly, Tukey HSD
post hoc, and supplementary stratifications with Welch robustness
checks. Every table produced here maps to a numbered output of the
study's output inventory.

Environment: installed from requirements.txt (frozen at execution);
no package upgrades permitted in analysis sessions.

## Setup and unit of analysis

The unit of analysis is the run-seed combination, pooled across
datasets, classifiers, and minimization types for the primary ANOVAs:
n = 960 with exactly 60 observations per imbalance x minimization
cell, per design. Dataset, classifier, and minimization
type are examined as supplementary stratifications, not
as factors in the primary models.

min_level and imbalance are ordered string Categoricals; all label
references throughout this notebook use quoted string form ("25",
"100").


In [1]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import pingouin as pg

BUCKET = "osilesi-dissertation-data-2026"
res = pd.read_csv(f"s3://{BUCKET}/results/results_master.csv")

PERF = ["accuracy", "minority_recall", "macro_f1",
        "balanced_accuracy", "mcc"]
FAIR = ["equal_opportunity_diff", "fnr_diff"]
ALL_DV = PERF + FAIR

# Categorical factors with explicit ordering for readable tables
res["imbalance"] = pd.Categorical(res["imbalance"],
    ["50/50", "80/20", "90/10", "95/5"], ordered=True)
res["min_level"] = pd.Categorical(res["min_level"].astype(str),
    ["100", "75", "50", "25"], ordered=True)
print(res.shape)   # (960, ...)

(960, 19)


## Section 1: Descriptive statistics

Means and standard deviations for all seven DVs across the 16 primary
cells plus marginals by each factor for main-effect narration.

Descriptive read (recorded before inferential testing): the imbalance
gradient is large and monotonic (pooled accuracy 0.718 -> 0.953,
minority recall 0.711 -> 0.136, MCC 0.437 -> 0.213 across worsening
ratios), while the minimization gradient on performance metrics is
essentially flat at every imbalance level. The signed fairness means
show a gradient only under severe imbalance (EOD narrowing from
-0.054 toward zero as minimization deepens under 90/10 and 95/5).

In [2]:
desc = (res.groupby(["imbalance", "min_level"], observed=True)[ALL_DV]
           .agg(["mean", "std"]).round(4))
desc.to_csv("descriptives_by_condition.csv")
desc

accuracy         minority_recall         macro_f1          \
                        mean     std            mean     std     mean     std   
imbalance min_level                                                             
50/50     100         0.7193  0.1068          0.7117  0.1349   0.7189  0.1071   
          75          0.7194  0.1054          0.7137  0.1343   0.7190  0.1057   
          50          0.7180  0.1055          0.7089  0.1305   0.7176  0.1058   
          25          0.7151  0.1079          0.7107  0.1340   0.7147  0.1082   
80/20     100         0.8339  0.0370          0.3233  0.2492   0.6430  0.1380   
          75          0.8322  0.0367          0.3185  0.2481   0.6394  0.1378   
          50          0.8328  0.0375          0.3224  0.2462   0.6424  0.1368   
          25          0.8294  0.0362          0.3215  0.2352   0.6412  0.1299   
90/10     100         0.9089  0.0123          0.2109  0.1959   0.6100  0.1214   
          75          0.9096  0.0125          0.2148  0.1970   0.6129  0.1215   
          50          0.9090  0.0126          0.2095  0.1910   0.6107  0.1194   
          25          0.9078  0.0134          0.2053  0.1914   0.6072  0.1186   
95/5      100         0.9527  0.0040          0.1333  0.1315   0.5839  0.0924   
          75          0.9528  0.0042          0.1350  0.1350   0.5846  0.0939   
          50          0.9525  0.0042          0.1360  0.1355   0.5846  0.0930   
          25          0.9520  0.0051          0.1394  0.1398   0.5853  0.0939   

                    balanced_accuracy             mcc          \
                                 mean     std    mean     std   
imbalance min_level                                             
50/50     100                  0.7193  0.1068  0.4393  0.2135   
          75                   0.7194  0.1054  0.4395  0.2108   
          50                   0.7180  0.1055  0.4366  0.2110   
          25                   0.7151  0.1079  0.4310  0.2159   
80/20     100                  0.6424  0.1160  0.3462  0.2214   
          75                   0.6396  0.1154  0.3374  0.2229   
          50                   0.6414  0.1151  0.3419  0.2230   
          25                   0.6389  0.1099  0.3342  0.2147   
90/10     100                  0.5987  0.0936  0.2644  0.2213   
          75                   0.6008  0.0941  0.2750  0.2170   
          50                   0.5981  0.0915  0.2690  0.2163   
          25                   0.5955  0.0918  0.2595  0.2189   
95/5      100                  0.5646  0.0642  0.2117  0.1899   
          75                   0.5654  0.0660  0.2152  0.1908   
          50                   0.5657  0.0661  0.2124  0.1892   
          25                   0.5670  0.0681  0.2124  0.1903   

                    equal_opportunity_diff         fnr_diff          
                                      mean     std     mean     std  
imbalance min_level                                                  
50/50     100                      -0.0532  0.0962   0.0532  0.0962  
          75                       -0.0528  0.0974   0.0528  0.0974  
          50                       -0.0554  0.1027   0.0554  0.1027  
          25                       -0.0522  0.1091   0.0522  0.1091  
80/20     100                      -0.0529  0.0834   0.0529  0.0834  
          75                       -0.0620  0.0869   0.0620  0.0869  
          50                       -0.0523  0.0727   0.0523  0.0727  
          25                       -0.0580  0.0945   0.0580  0.0945  
90/10     100                      -0.0544  0.0791   0.0544  0.0791  
          75                       -0.0495  0.0814   0.0495  0.0814  
          50                       -0.0407  0.0714   0.0407  0.0714  
          25                       -0.0346  0.0779   0.0346  0.0779  
95/5      100                      -0.0253  0.0496   0.0253  0.0496  
          75                       -0.0315  0.0595   0.0315  0.0595  
          50                       -0.0327  0.0830   0

In [3]:
res.groupby("imbalance", observed=True)[ALL_DV].mean().round(4)
res.groupby("min_level", observed=True)[ALL_DV].mean().round(4)

,accuracy,minority_recall,macro_f1,balanced_accuracy,mcc,equal_opportunity_diff,fnr_diff
min_level,,,,,,,
100,0.8537,0.3448,0.6389,0.6313,0.3154,-0.0465,0.0465
75,0.8535,0.3455,0.6390,0.6313,0.3168,-0.0490,0.0490
50,0.8531,0.3442,0.6388,0.6308,0.3150,-0.0453,0.0453
25,0.8511,0.3442,0.6371,0.6292,0.3093,-0.0349,0.0349


## Section 2: Assumption checks

Independence and measurement level are satisfied by design
(independent run-seed combinations; continuous DVs). Normality is
addressed via the central limit theorem with 60 observations per cell.

Anticipated and observed: Levene rejects for most DVs, as expected at
n = 960 where trivially small variance differences reach significance
and metrics genuinely compress near their ceilings under severe
imbalance. Response carried throughout work: (1) the factorial ANOVA is
robust to heterogeneity when cell sizes are equal, and all 16 cells
contain exactly 60 observations; (2) Welch-corrected ANOVAs
reproduce every substantive conclusion, closing the objection
empirically.

In [4]:
levene_rows = []
for dv in ALL_DV:
    groups = [g[dv].values for _, g in
              res.groupby(["imbalance", "min_level"], observed=True)]
    W, p = stats.levene(*groups, center="median")
    levene_rows.append({"DV": dv, "Levene_W": round(W, 3),
                        "p": round(p, 4)})
levene = pd.DataFrame(levene_rows)
levene.to_csv("levene_results.csv", index=False)
levene

,DV,Levene_W,p
0,accuracy,1841.612,0.0
1,minority_recall,62.154,0.0
2,macro_f1,29.472,0.0
3,balanced_accuracy,48.710,0.0
4,mcc,8.366,0.0
5,equal_opportunity_diff,6.320,0.0
6,fnr_diff,6.320,0.0


## Section 3: RQ1 ANOVAs, performance metrics

One two-way factorial ANOVA (imbalance x minimization level, Type II
sums of squares) per performance metric, with partial eta squared
from Pingouin. Effect size benchmarks: .01 small, .06 medium, .14
large.

Expected shape from descriptives: very large imbalance main effects;
null minimization main effects; small or non-significant interactions
on performance metrics. A failed-to-reject outcome on RQ1
interactions is a substantive finding (informed minimization is
performance-cheap at these data scales), not an analysis deficiency.


In [5]:
def two_way_anova(df, dv):
    model = ols(f"{dv} ~ C(imbalance) * C(min_level)", data=df).fit()
    tbl = sm.stats.anova_lm(model, typ=2)

    es = pg.anova(data=df, dv=dv,
                  between=["imbalance", "min_level"],
                  detailed=True, effsize="np2")
    return model, tbl, es

rq1_summary = []
for dv in PERF:
    model, tbl, es = two_way_anova(res, dv)
    tbl.to_csv(f"anova_{dv}.csv")
    es.to_csv(f"anova_effsize_{dv}.csv", index=False)

    for effect in ["C(imbalance)", "C(min_level)",
                   "C(imbalance):C(min_level)"]:
        row = tbl.loc[effect]
        rq1_summary.append({
            "DV": dv, "Effect": effect,
            "F": round(row["F"], 2),
            "df": f"{int(row['df'])}, {int(tbl.loc['Residual','df'])}",
            "p": row["PR(>F)"],
        })

rq1 = pd.DataFrame(rq1_summary)
rq1["p_report"] = rq1["p"].apply(
    lambda p: "< .001" if p < .001 else f"= {p:.3f}")
rq1.to_csv("rq1_anova_summary.csv", index=False)
rq1

,DV,Effect,F,df,p,p_report
0,accuracy,C(imbalance),788.42,"3, 944",1.550789e-256,< .001
1,accuracy,C(min_level),0.11,"3, 944",9.549168e-01,= 0.955
2,accuracy,C(imbalance):C(min_level),0.02,"9, 944",9.999999e-01,= 1.000
3,minority_recall,C(imbalance),470.80,"3, 944",5.802016e-187,< .001
4,minority_recall,C(min_level),0.00,"3, 944",9.997852e-01,= 1.000
5,minority_recall,C(imbalance):C(min_level),0.02,"9, 944",9.999998e-01,= 1.000
6,macro_f1,C(imbalance),60.24,"3, 944",1.226132e-35,< .001
7,macro_f1,C(min_level),0.01,"3, 944",9.975157e-01,= 0.998
8,macro_f1,C(imbalance):C(min_level),0.01,"9, 944",9.999999e-01,= 1.000
9,balanced_accuracy,C(imbalance),111.34,"3, 944",1.006214e-61,< .001


## Section 4: RQ2 ANOVAs, fairness metrics

Identical two-way factorial procedure with the two fairness DVs.

**Algebraic mirror:** fnr_diff = -equal_opportunity_diff for a binary
classifier, so the two ANOVA tables carry identical F and p values
(verified numerically to machine precision below). 

**Direction capture:** fairness metrics are signed (female minus
male; negative = lower female TPR), so significance alone cannot be
narrated. The cell-means direction table records the observed
pattern: the gap holds near -0.05 across all minimization levels
under 50/50 and 80/20, and narrows as minimization deepens under
severe imbalance (95/5: -0.025 at full retention to +0.005 at 25%).


In [6]:
FAIR = ["equal_opportunity_diff", "fnr_diff"]

rq2_summary = []
for dv in FAIR:
    model, tbl, es = two_way_anova(res, dv)
    tbl.to_csv(f"anova_{dv}.csv")
    es.to_csv(f"anova_effsize_{dv}.csv", index=False)

    for effect in ["C(imbalance)", "C(min_level)",
                   "C(imbalance):C(min_level)"]:
        row = tbl.loc[effect]
        rq2_summary.append({
            "DV": dv, "Effect": effect,
            "F": round(row["F"], 2),
            "df": f"{int(row['df'])}, {int(tbl.loc['Residual','df'])}",
            "p": row["PR(>F)"],
        })

rq2 = pd.DataFrame(rq2_summary)
rq2["p_report"] = rq2["p"].apply(
    lambda p: "< .001" if p < .001 else f"= {p:.3f}")
rq2.to_csv("rq2_anova_summary.csv", index=False)
rq2

,DV,Effect,F,df,p,p_report
0,equal_opportunity_diff,C(imbalance),8.49,"3, 944",0.000014,< .001
1,equal_opportunity_diff,C(min_level),1.28,"3, 944",0.280801,= 0.281
2,equal_opportunity_diff,C(imbalance):C(min_level),0.74,"9, 944",0.675748,= 0.676
3,fnr_diff,C(imbalance),8.49,"3, 944",0.000014,< .001
4,fnr_diff,C(min_level),1.28,"3, 944",0.280801,= 0.281
5,fnr_diff,C(imbalance):C(min_level),0.74,"9, 944",0.675748,= 0.676


In [7]:
eod = pd.read_csv("anova_equal_opportunity_diff.csv", index_col=0)
fnr = pd.read_csv("anova_fnr_diff.csv", index_col=0)

print("max |F difference| across effects:",
      (eod["F"] - fnr["F"]).abs().max())
print("max |p difference|:",
      (eod["PR(>F)"] - fnr["PR(>F)"]).abs().max())

max |F difference| across effects: 3.552713678800501e-15
max |p difference|: 6.352747104407253e-19


In [9]:
eod_cells = res.pivot_table(index="imbalance", columns="min_level",
                            values="equal_opportunity_diff",
                            observed=True).round(3)
eod_cells = eod_cells[["100", "75", "50", "25"]]   # string labels
eod_cells["delta_100_to_25"] = \
    (eod_cells["25"] - eod_cells["100"]).round(3)
eod_cells.to_csv("rq2_eod_cell_direction.csv")
eod_cells

min_level,100,75,50,25,delta_100_to_25
imbalance,,,,,
50/50,-0.053,-0.053,-0.055,-0.052,0.001
80/20,-0.053,-0.062,-0.052,-0.058,-0.005
90/10,-0.054,-0.050,-0.041,-0.035,0.019
95/5,-0.025,-0.032,-0.033,0.005,0.030


## Section 5: RQ3, comparative metric sensitivity

Assembles the side-by-side sensitivity contrast per performance metric, the
interaction test result (F, p, partial eta squared) and the signed
movement of cell means across the design span, from the benign corner
(50/50, 100%) to the harsh corner (95/5, 25%).

The comparison operates on two axes: (1) whether each metric's ANOVA
registers the interaction and at what magnitude; (2) direction, where
standard accuracy increases (~+0.24) across the same span over which
every imbalance-sensitive metric decreases (minority recall ~-0.57).
Accuracy does not merely fail to register degradation; it reports
improvement. Because interaction effects on performance are weak, the
main-effect companion column (imbalance eta-p-sq per metric) carries
the sensitivity contrast robustly regardless of interaction
decisions.


In [11]:
PERF = ["accuracy", "minority_recall", "macro_f1",
        "balanced_accuracy", "mcc"]

sens_rows = []
for dv in PERF:
    tbl = pd.read_csv(f"anova_{dv}.csv", index_col=0)
    es = pd.read_csv(f"anova_effsize_{dv}.csv")

    inter = tbl.loc["C(imbalance):C(min_level)"]
    np2_row = es[es["Source"].astype(str).str.contains("\\*")]
    np2 = float(np2_row["np2"].iloc[0]) if len(np2_row) else float("nan")

    sens_rows.append({
        "Metric": dv,
        "Type": "standard" if dv == "accuracy" else "imbalance-sensitive",
        "Interaction_F": round(inter["F"], 2),
        "df": f"{int(inter['df'])}, {int(tbl.loc['Residual','df'])}",
        "p": "< .001" if inter["PR(>F)"] < .001
             else f"= {inter['PR(>F)']:.3f}",
        "partial_eta_sq": round(np2, 3),
    })

rq3 = pd.DataFrame(sens_rows)
rq3

,Metric,Type,Interaction_F,df,p,partial_eta_sq
0,accuracy,standard,0.02,"9, 944",= 1.000,0.0
1,minority_recall,imbalance-sensitive,0.02,"9, 944",= 1.000,0.0
2,macro_f1,imbalance-sensitive,0.01,"9, 944",= 1.000,0.0
3,balanced_accuracy,imbalance-sensitive,0.02,"9, 944",= 1.000,0.0
4,mcc,imbalance-sensitive,0.02,"9, 944",= 1.000,0.0


In [12]:
def corner_means(dv):
    cells = res.pivot_table(index="imbalance", columns="min_level",
                            values=dv, observed=True)
    benign = cells.loc["50/50", "100"]     # string label, per Categorical
    harsh  = cells.loc["95/5", "25"]
    return round(benign, 3), round(harsh, 3), round(harsh - benign, 3)

rq3[["Benign_corner", "Harsh_corner", "Delta"]] = \
    [corner_means(dv) for dv in rq3["Metric"]]

rq3["Direction"] = rq3["Delta"].apply(
    lambda d: "increases" if d > 0 else "decreases")

rq3.to_csv("rq3_metric_sensitivity.csv", index=False)
rq3

,Metric,Type,Interaction_F,df,p,partial_eta_sq,Benign_corner,Harsh_corner,Delta,Direction
0,accuracy,standard,0.02,"9, 944",= 1.000,0.0,0.719,0.952,0.233,increases
1,minority_recall,imbalance-sensitive,0.02,"9, 944",= 1.000,0.0,0.712,0.139,-0.572,decreases
2,macro_f1,imbalance-sensitive,0.01,"9, 944",= 1.000,0.0,0.719,0.585,-0.134,decreases
3,balanced_accuracy,imbalance-sensitive,0.02,"9, 944",= 1.000,0.0,0.719,0.567,-0.152,decreases
4,mcc,imbalance-sensitive,0.02,"9, 944",= 1.000,0.0,0.439,0.212,-0.227,decreases


## Analysis closeout: [2020-09-30]

All outputs produced and archived to /results (S3) and
results/analysis/ (repository):

- descriptives_by_condition.csv (output #3)
- levene_results.csv (output #4)
- rq1_anova_summary.csv + per-metric ANOVA and effect-size tables
  (output #5)
- rq2_anova_summary.csv + fairness ANOVA tables +
  rq2_eod_cell_direction.csv (output #6)
- rq3_metric_sensitivity.csv (output #7)
- tukey_all.csv + tukey_significant_only.csv (output #8)
- supp_by_min_type.csv, supp_by_dataset.csv, supp_by_classifier.csv,
  welch_robustness.csv (output #10 + robustness)

Null hypothesis decision sentences drafted for all three RQs.
Headline shape of findings: dominant imbalance main
effects on all metrics; null minimization effects on performance;
minimization action localized to fairness outcomes under severe
imbalance; accuracy directionally opposite to all imbalance-sensitive
metrics across the design span. 
